In [19]:
import numpy as np
import math

def kramer(A, f):
    N = len(A)
    main_det = np.linalg.det(A)
    res = np.zeros(N)
    for j in range(N):
        A_j = A.copy() # матрица с замененным j-ым столбцом на столбец свободных членов
        for i in range(N):
            A_j[i][j] = f[i]
        det_i = np.linalg.det(A_j)
        res[j] = det_i / main_det
    return res

def inv_matrix(A, f):
    A_inv = np.linalg.inv(A)
    return A_inv @ f
    
N = 3
A = np.random.randint(low = 0, high = 100, size = (N, N))
f = np.random.randint(low = 0, high = 100, size = N)

for i in range(N):
    space_or_equal = "=" if (i == N // 2) else " "
    print(f"{A[i]}[x{i+1}] {space_or_equal} [{f[i]}]")


if np.linalg.det(A) == 0:
    print("Cистема имеет 0 или ∞ кол-во решений") 
else:
    print("\nРешение по правилу Крамера:")
    print(kramer(A, f))

    print("\nРешение через обратную матрицу:")
    print(inv_matrix(A, f))
    

[95 61 37][x1]   [93]
[66 37 66][x2] = [44]
[93 60  0][x3]   [95]

Решение по правилу Крамера:
[-1.04385512  3.20130877 -0.08415131]

Решение через обратную матрицу:
[-1.04385512  3.20130877 -0.08415131]


In [4]:
import numpy as np

# Возвращает номер строки с максимальным элементом
def find_main_el(A, k):
    max_val = abs(A[k][k])
    max_row = k
    
    for i in range(k + 1, len(A)):
        if abs(A[i][k]) > max_val:
            max_val = abs(A[i][k])
            max_row = i
    
    return max_row if max_val > 1e-10 else None

# Выводит расширенную матрицу
def print_m(A):
    for i in range(len(A)):
        print("[", *A[i][:-1], "|", A[i][-1], "]")
        
def gauss(A):
    N = len(A)
    A = A.copy().astype(float)
    
    # Прямой ход
    for k in range(N - 1):        
        pivot_row = find_main_el(A, k)
        
        if pivot_row is None:
            print("Система вырождена")
            return None
        
        if pivot_row != k:
            A[[k, pivot_row]] = A[[pivot_row, k]]
                
        for i in range(k + 1, N):
            if A[k][k] != 0:
                factor = A[i][k] / A[k][k]
                A[i][k:] -= factor * A[k][k:]
    
    # Обратный ход
    res = np.zeros(N)
    for i in range(N - 1, -1, -1):
        if abs(A[i][i]) < 1e-10:
            print("Система вырождена")
            return None
        sum_ax = sum(A[i][j] * res[j] for j in range(i + 1, N))
        res[i] = (A[i][N] - sum_ax) / A[i][i]
    
    return res

A = np.array([[62, 89, 96, 88],
              [68, 61, 28, 75],
              [85, 16, 44, 29]])

print("Система уравнений:")
print_m(A)

print("Решение с помощью метода Гаусса с выбором главного члена")
print(gauss(A))

Система уравнений:
[ 62 89 96 | 88 ]
[ 68 61 28 | 75 ]
[ 85 16 44 | 29 ]
Решение с помощью метода Гаусса с выбором главного члена
[ 0.25647263  1.04250391 -0.21545991]


In [5]:
import numpy as np

def print_m(A):
    for i in range(len(A)):
        print("[", *A[i][:-1], "|", A[i][-1], "]")

def gauss_qr(A):
    N = len(A)
    A_mat = A[:, :-1].copy().astype(float)
    f = A[:, -1].copy().astype(float)
    
    Q, R = np.linalg.qr(A_mat)
    f_transform = Q.T @ f
    
    res = np.zeros(N)
    for i in range(N - 1, -1, -1):
        sum_ax = sum(R[i][j] * res[j] for j in range(i + 1, N))
        res[i] = (f_transform[i] - sum_ax) / R[i][i]
    
    return res

A = np.array([[62, 89, 96, 88],
              [68, 61, 28, 75],
              [85, 16, 44, 29]])

print("Система уравнений:")
print_m(A)

print("Решение методом Гаусса с помощью np.linalg.qr()")
print(gauss_qr(A))

Система уравнений:
[ 62 89 96 | 88 ]
[ 68 61 28 | 75 ]
[ 85 16 44 | 29 ]
Решение методом Гаусса с помощью np.linalg.qr()
[ 0.25647263  1.04250391 -0.21545991]


In [33]:
import numpy as np

n = 20
A = np.random.rand(n, n)
X = np.random.rand(n)

# для увеличения числа обусловленности надо создать почти линейную записимость между стр/столб
A[:, 0] = A[:, 1] + 0.1

cond = np.linalg.cond(A)
print(f"cond(A) = {cond}")

f = A @ X
A_extended = np.column_stack([A, f]) 

X_kramer = kramer(A, f)
X_inv = inv_matrix(A, f)
X_gauss_no_main_el = gauss_qr(A_extended)
X_gauss_with_main_el = gauss(A_extended)

def rel_error(x, x_true):
    return np.linalg.norm(x - x_true) / np.linalg.norm(x_true)

err_kramer = rel_error(X_kramer, X)
err_inv = rel_error(X_inv, X)
err_with_main_el = rel_error(X_gauss_with_main_el, X)
err_no_main_el = rel_error(X_gauss_no_main_el, X)

print("\nОтносительные ошибки")
print(f"Крамер:                       {err_kramer:.2e}")
print(f"Обратная матрица:             {err_inv:.2e}")
print(f"Гаусс с выбором главного эл:  {err_with_main_el:.2e}")
print(f"Гаусс без выбора главного эл: {err_no_main_el:.2e}")

cond(A) = 4493.956283528275

Относительные ошибки
Крамер:                       7.78e-14
Обратная матрица:             3.94e-13
Гаусс с выбором главного эл:  3.13e-14
Гаусс без выбора главного эл: 8.48e-14
